# Configurer la connexion

In [4]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
load_dotenv()  # installer .env
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")
DATABASE_URL = f"postgresql://{user}:{password}@{host}:{port}/{db}"
engine = create_engine(DATABASE_URL)
print("conexion ok")

conexion ok


# Séparer le dataset CSV selon les tables normalisées

In [5]:
import pandas as pd
df = pd.read_csv("c:/breif_sql_banc/data/financecore_clean.csv")
#table client
client = df[['client_id', 'score_credit_client', 'segment_client', 'agence','categorie_risque']].drop_duplicates()
client = client.rename(columns={
    'score_credit_client': 'score_credit',
    'segment_client': 'segment_id',
    'agence': 'agence_id'
   })
#table segment
segment = df[['segment_client']].drop_duplicates()
segment = segment.rename(columns={
    'segment_client': 'nom_segment'
})
#table compte
compte = df[['client_id', 'solde_avant']].drop_duplicates()
compte=compte.rename(columns={
    'solde_avant':'solde'
})
#table produit
produit= df[['produit','categorie']].drop_duplicates()
produit=produit.rename(columns={
    'produit':'nom_produit'
})
#table agence
agence=df[['agence']].drop_duplicates()
agence= agence.rename(columns={
'agence':'agence_nom'
})
##table periode
period=df[['annee', 'mois', 'trimestre', 'jour_semaine']].drop_duplicates()
###IDs
segment['segment_id'] = range(1, len(segment) + 1)
produit['produit_id'] = range(1, len(produit) + 1)
agence['agence_id'] = range(1, len(agence) + 1)
compte['compte_id'] = range(1, len(compte) + 1)
period['period_id'] = range(1, len(period) + 1)
###MERGE
df = df.merge(segment, left_on='segment_client', right_on='nom_segment', how='left')
df = df.merge(produit, left_on='produit', right_on='nom_produit', how='left')
df = df.merge(agence, left_on='agence', right_on='agence_nom', how='left')
df = df.merge(compte, on='client_id', how='left')
df = df.merge(
    period,
    on=['annee', 'mois', 'trimestre', 'jour_semaine'],
    how='left'
)
#table transaction
transaction = df[['transaction_id', 'compte_id', 'produit_id','period_id','client_id',
                  'date_transaction', 'montant',
                  'type_operation', 'statut', 'devise','montant_eur']].drop_duplicates()

# Insérer les données 

In [6]:
# table client
client.to_sql("client", engine, if_exists="append", index=False)
#table segment
segment.to_sql("segment",engine,if_exists="append",index=False)
#table agence
agence.to_sql("agence",engine,if_exists="append",index=False)
#table produit
produit.to_sql("produit",engine, if_exists="append",index=False)
#table compte
compte.to_sql("compte",engine,if_exists="append",index=False)
#table transaction
transaction.to_sql("transaction",engine,if_exists="append",index=False)
#table period
period.to_sql("period",engine,if_exists="append",index=False)

900